In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import logging
logging.getLogger('matplotlib.font_manager').setLevel(level=logging.CRITICAL)

In [ ]:
from matplotlib import pyplot as plt
from tqdm import tqdm
import itertools
import os
import numpy as np
import scipy.sparse
import pandas as pd
import re
import tifffile
import xmltodict
import importlib.util
import sys
import scanpy as sc


In [ ]:
import importlib.util
import sys

def lazy_import(module_name, path_to_file):
    spec = importlib.util.spec_from_file_location(module_name,path_to_file)
    foo = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = foo
    spec.loader.exec_module(foo)
    return foo

utils_dir = "../../utils"
general_utils =  lazy_import("general_utils",os.path.join(utils_dir, "general_utils.py"))

from general_utils import ismember, grep, grep_exclude


In [ ]:
tenx_data_dir = "../xenium_rawdata/"


An example of the xenium_file_manifest_tier_classification.csv is provided in the repository. The code below can be used to generate it from scratch based on the contents of the xenium_rawdata directory. In addition, you can add columns to the .csv file to denote metadata to be included in the adata object, as well as tiered classification of samples. 

Sample tiers are used in subsequent scripts to embed subsets of the data

In [ ]:
file_database_path = '../support/xenium_file_manifest_tier_classification.csv'
if not os.path.exists(file_database_path):
    experiment_paths = [os.path.join(tenx_data_dir, x) for x in np.sort(os.listdir(tenx_data_dir))]
    region_paths = list(itertools.chain(*[[os.path.join(x, y) for y in np.sort(os.listdir(x))] for x in experiment_paths]))
    region_paths = grep_exclude("IF", region_paths)
    simplified_experiment_prefix = [re.findall("(\d+__Region_\d)", x)[0] for x in region_paths]
    simplified_experiment_prefix = [re.sub('__', '_', x) for x in simplified_experiment_prefix]
    file_database = pd.DataFrame({"slide_path": region_paths, "prefix": simplified_experiment_prefix})
    file_database.to_csv(file_database_path)
else:
    file_database = pd.read_csv(file_database_path, index_col = 0)

In [ ]:
file_database.index = list(range(file_database.shape[0]))

In [ ]:
output_path = '../xenium_preprocessing_outputs/adata_single_sample'
os.makedirs(output_path, exist_ok=True)

In [ ]:
for i in range(file_database.shape[0]):
    output_file = os.path.join(output_path, 'adata_%s.h5ad' % (file_database['prefix'][i]))
    if not os.path.exists(output_file):
        adata = compile_adata(file_database, i)
        sc.write(output_file, adata = adata)